# Instagram export — scan for personal data

This notebook runs **entirely inside your browser**. The Python interpreter was downloaded into this tab,
and the files you drag in are kept in your browser's local storage on your own computer. Nothing is sent to
any server. (Check it yourself: once the page has loaded, switch off Wi-Fi and everything still works.)

**Before you start** — get your data from Instagram: *Settings → Accounts Center → Your information and
permissions → Download your information*. Choose **JSON** format, **all time**, and **media quality: low**
(or deselect media). You will get an e-mail with a `.zip` file: download it, do **not** unzip it, and drag it
into the file browser on the left of this page.

Then run the cells below, top to bottom, with **Shift+Enter**.

In [ ]:
# 1. Find the export you uploaded (a .zip next to this notebook).
import os, sys, glob

if not os.path.exists("ig_pii_scan.py"):          # in case the notebook was opened from another folder
    for root, _dirs, files in os.walk("/drive"):
        if "ig_pii_scan.py" in files:
            os.chdir(root)
            break
sys.path.insert(0, os.getcwd())

exports = sorted(glob.glob("*.zip")) + sorted(d for d in os.listdir() if d.startswith("instagram-") and os.path.isdir(d))
if exports:
    EXPORT = exports[0]
    print("Export found:", EXPORT, f"({os.path.getsize(EXPORT) / 1e6:.1f} MB)" if os.path.isfile(EXPORT) else "")
    if len(exports) > 1:
        print("Several exports found; using the first one. Change EXPORT = ... to pick another:", exports)
else:
    EXPORT = None
    print("Nothing found yet: drag your instagram-....zip into the file browser on the left, then re-run this cell.")

In [ ]:
# 2. Scan it. The summary below shows masked values only, so it is safe to look at on a shared screen.
from ig_pii_scan import scan

report = scan(EXPORT)
print(report.summary())

**How to read the summary.** The first block counts what was found by category — `HIGH` means a direct
identifier (e-mail, phone number, IP address, coordinates, date of birth…), `MEDIUM` a name, username or
device/location detail, `REVIEW` something to look at by hand. "People referenced" separates *you* from
*other people*: the other people (your followers, the friends in your DMs, the contacts synced from your
phone) never agreed to be part of any dataset, and that is who the anonymisation step has to protect.
The second block lists the files, most sensitive first, with the action to take before anything is shared.

In [ ]:
# 3. Write the full report (with the raw values) into a folder, and pack it as one zip you can download.
import shutil

written = report.write("report")
shutil.make_archive("report", "zip", "report")
print("Written:")
for p in written:
    print("  ", p)
print("\nRight-click `report.zip` in the file browser on the left and choose Download to keep a copy on your computer.")

In [ ]:
# 4. Optional: look at the findings as a table. (The first import downloads pandas into the browser: that is code,
#    not your data. If it fails, skip this cell: the CSV files in report/ hold the same information.)
try:
    import pandas as pd
except ImportError:
    pd = None

if pd is None:
    print("pandas could not be loaded (no internet connection?). Skip this cell.")
else:
    df = report.to_dataframe()
    display(df.groupby(["severity", "category"]).size().rename("findings").reset_index()
              .sort_values(["severity", "findings"], ascending=[True, False]).reset_index(drop=True))
    display(df[df.severity == "high"][["category", "file", "key", "masked"]].head(20))

## What next

- The `report/pii_people.csv` file is the list of everyone the export mentions. The next notebook will use it to
  replace each person with a stable pseudonym (`user_0417`), drop the files marked **DROP**, strip location tags,
  and reduce other people's messages to timestamps and lengths — so that only an anonymised copy ever leaves
  your computer.
- Free text (captions, comments, DM content) is where automated detection is weakest: nothing being flagged in a
  message does not mean it is clean.

**Clean up when you are done:** right-click the zip, the `report/` folder and `report.zip` → *Delete*, or clear
this site's data in your browser settings.